## Configure API credentials
Provide your API key, channel, and region. This cell sets the session environment and can also persist a config file at `~/.config/mips/atlaspy/config.json` for reuse.

In [ ]:
# Configure API credentials for ATLAS Explorer
# Fill in your values and run this cell.
# - Sets MIPS_ATLAS_CONFIG for this session
# - Optionally writes ~/.config/mips/atlaspy/config.json for reuse

from pathlib import Path
import os, json

# REQUIRED: set these
ae_apikey = ""  # e.g., "your-api-key"
ae_channel = ""  # e.g., "development"
ae_region = ""  # e.g., "us-west-2" or vendor-provided region

# OPTIONAL: persist to config file for future runs
persist_to_file = True

if not ae_apikey or not ae_channel or not ae_region:
    print("Please set ae_apikey, ae_channel, and ae_region above, then re-run.")
else:
    os.environ["MIPS_ATLAS_CONFIG"] = f"{ae_apikey}:{ae_channel}:{ae_region}"
    print("Session env set: MIPS_ATLAS_CONFIG")

    if persist_to_file:
        cfg_path = Path.home() / ".config/mips/atlaspy/config.json"
        cfg_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cfg_path, "w") as f:
            json.dump({"apikey": ae_apikey, "channel": ae_channel, "region": ae_region}, f, indent=2)
        print(f"Wrote config file: {cfg_path}")
    else:
        print("Skipping config file write (persist_to_file=False)")

# ATLAS Explorer: Single-core experiment (Notebook)

Run a single-core experiment and optionally export a report.

## Prerequisites
- Configure credentials once: run the CLI configurator or set `MIPS_ATLAS_CONFIG`.
- Example CLI: `uv run atlasexplorer/atlasexplorer.py configure`.
- Ensure example ELF exists (default uses files in `resources/`).

In [ ]:
# Settings — adjust as needed
elf = "resources/mandelbrot_rv64_O0.elf"
expdir = "myexperiments"
core = "I8500_(1_thread)"
channel = "development"
apikey = None   # or a string
region = None   # or a string
verbose = False

# Export options: None, 'json', 'markdown', 'html', 'rich-html', or 'zip'
export = None
# Optional explicit output path (defaults to a sensible name near summary.json)
out = None

In [ ]:
# Environment setup
import os, locale
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
locale.setlocale(locale.LC_ALL, "")

config_env = os.environ.get("MIPS_ATLAS_CONFIG")
home_dir = os.path.expanduser("~")
config_file = os.path.join(home_dir, ".config", "mips", "atlaspy", "config.json")
if not apikey and not config_env and not os.path.exists(config_file):
    print("Tip: run 'uv run atlasexplorer/atlasexplorer.py configure' or set MIPS_ATLAS_CONFIG.")

In [ ]:
# Run the experiment
from atlasexplorer.atlasexplorer import AtlasExplorer, Experiment

aeinst = AtlasExplorer(apikey, channel, region, verbose=verbose)
experiment = Experiment(expdir, aeinst, verbose=verbose)
experiment.addWorkload(os.path.abspath(elf))
experiment.setCore(core)
experiment.run()

# Find newest summary.json
exp_path = Path(expdir)
summary_candidates = list(exp_path.rglob("summary/summary.json"))
if not summary_candidates:
    raise FileNotFoundError("No summary.json found under experiment directory")
summary_candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
summary_path = summary_candidates[0]
print(f"Produced summary: {summary_path}")

In [ ]:
# Optional export + quick KPI
if export:
    from atlasexplorer.reporting.parser import parse_summary_json
    from atlasexplorer.reporting.derive import apply_derivations
    from atlasexplorer.reporting.thresholds import apply_thresholds
    from atlasexplorer.reporting.export import (
        export_json, export_markdown, export_html, export_rich_html, export_zip
    )

    report = parse_summary_json(str(summary_path))
    report = apply_derivations(report)
    report = apply_thresholds(report)

    default_dir = summary_path.parent
    default_names = {
        'json': 'report.json',
        'markdown': 'report.md',
        'html': 'report.html',
        'rich-html': 'report_rich.html',
        'zip': 'report_bundle.zip',
    }
    out_path = Path(out) if out else (default_dir / default_names[export])
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if export == 'json':
        export_json(report, str(out_path))
    elif export == 'markdown':
        export_markdown(report, str(out_path))
    elif export == 'html':
        export_html(report, str(out_path))
    elif export == 'rich-html':
        export_rich_html(report, str(out_path))
    elif export == 'zip':
        export_zip(report, str(out_path), rich=True)
    print(f"Export written to: {out_path}")
else:
    try:
        total_cycles = experiment.getSummary().getTotalCycles()
        print(f"Total Cycles: {total_cycles}")
    except Exception:
        pass

In [ ]:
# Quick ad-hoc analysis as a DataFrame
from atlasexplorer.reporting.parser import parse_summary_json
from atlasexplorer.reporting.derive import apply_derivations
import pandas as pd

report = apply_derivations(parse_summary_json(str(summary_path)))
df = pd.DataFrame([m.model_dump() for m in report.metrics])
df.head()